In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day04" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w5" / "day04"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"

MODELS = ROOT.parent / "models"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


In [2]:
from app.core.exceptions import ValidationFailed
from app.rag.local_parsers import parse_local

scanned = FORMATS / "sample_scanned.pdf"
normal = SAMPLES / "DOC-HR-014_국내출장_여비_규정_v2.0.pdf" 

print(scanned.name)

try:
    parse_local(str(scanned))
except ValidationFailed as e:
    print(f" ValidationFailed : {e}")
    print(f" detail : {e.detail}")
print()

doc = parse_local(str(normal))
print(normal.name)
print(f"블록 {len(doc.blocks)}, 표 {doc.table_count}, 페이지 {doc.page_count}")

sample_scanned.pdf
15:45:26 INFO     app.rag.local_parsers: 텍스트 없음 (스캔본으로 보임): sample_scanned.pdf p.1
 ValidationFailed : 문서에서 글자를 찾지 못했습니다: sample_scanned.pdf
 detail : 스캔본(이미지)일 수 있습니다. .env 의 UPSTAGE_PARSE_OCR 를 force 로 두고 실동작 모드로 다시 적재해 보세요

DOC-HR-014_국내출장_여비_규정_v2.0.pdf
블록 16, 표 0, 페이지 16


In [3]:
import pdfplumber
import pypdf


def pypdf_stat(path):
    reader = pypdf.PdfReader(str(path))
    return len(reader.pages), sum(len(pg.extract_text() or "") for pg in reader.pages)


def plumber_stat(path):
    with pdfplumber.open(str(path)) as pdf:
        chars = sum(len(pg.extract_text() or "") for pg in pdf.pages)
        tables = sum(len(pg.extract_tables()) for pg in pdf.pages)
        return len(pdf.pages), chars, tables 

print("scanned!")
a_pages, a_chars = pypdf_stat(scanned)
print(f" pypdf 페이지 {a_pages}, 글자 {a_chars}")
b_pages, b_chars, b_tables = plumber_stat(scanned)
print(f" plumber 페이지 {b_pages}, 글자 {b_chars}, 표 {b_tables}")

print()
print("normal!")
c_pages, _ = pypdf_stat(normal)
print(f" pypdf  페이지 {c_pages}, 표 {doc.table_count}")
d_pages, _, d_tables = plumber_stat(normal)
print(f" plumber  페이지 {d_pages}, 표 {d_tables}")

scanned!
 pypdf 페이지 1, 글자 0
 plumber 페이지 1, 글자 0, 표 0

normal!
 pypdf  페이지 16, 표 0
 plumber  페이지 16, 표 7
